# 07 — Missing Value Analysis

## Objective

This notebook evaluates missingness and temporal coverage in the prepared dengue, weather, and population datasets before imputation experiments and dataset integration.

The objectives are to:

1. verify missing values in each prepared dataset;
2. verify expected district-month coverage;
3. distinguish ordinary missing values from structural source-coverage limitations;
4. identify incomplete weather boundary months;
5. quantify the population coverage gap for 2010–2013;
6. construct a common district-month reference grid for systematic missingness analysis;
7. determine which variables are suitable for controlled missing-value experiments.

No missing values are imputed in this notebook.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
print("Project root:", PROJECT_ROOT)
print("Interim directory:", INTERIM_DIR)

Project root: d:\Dengue_Project
Interim directory: d:\Dengue_Project\data\interim


In [3]:
interim_files = sorted(
    INTERIM_DIR.glob("*.csv")
)
for file in interim_files:
    print(file.name)

dengue_monthly_enddate_baseline_2010_2024.csv
dengue_monthly_proportional_2010_2024.csv
dengue_weekly_geography_standardized.csv
geography_mapping.csv
population_geography_standardized.csv
population_monthly_prepared_2014_2024.csv
weather_monthly_prepared_2010_2024.csv
weather_weekly_geography_standardized.csv


Identify weather and population files

In [4]:
weather_path = (
    INTERIM_DIR /
    "weather_monthly_prepared_2010_2024.csv"
)
population_path = (
    INTERIM_DIR /
    "population_monthly_prepared_2014_2024.csv"
)
print("Weather exists:", weather_path.exists())
print("Population exists:", population_path.exists())

Weather exists: True
Population exists: True


Find the dengue candidates

In [5]:
dengue_candidates = sorted(
    INTERIM_DIR.glob(
        "*dengue*monthly*.csv"
    )
)
for file in dengue_candidates:
    print(file.name)

dengue_monthly_enddate_baseline_2010_2024.csv
dengue_monthly_proportional_2010_2024.csv


In [6]:
for file in dengue_candidates:
    temp = pd.read_csv(file)
    print("\nFILE:", file.name)
    print("Shape:", temp.shape)
    print("Columns:", temp.columns.tolist())


FILE: dengue_monthly_enddate_baseline_2010_2024.csv
Shape: (4500, 5)
Columns: ['date', 'year', 'month_number', 'administrative_district', 'dengue_cases']

FILE: dengue_monthly_proportional_2010_2024.csv
Shape: (4500, 5)
Columns: ['date', 'year', 'month_number', 'administrative_district', 'dengue_cases']


# Select the prepared dengue dataset

In [7]:
dengue_path = (
    INTERIM_DIR /
    "dengue_monthly_proportional_2010_2024.csv"
)
print(
    "Selected dengue file:",
    dengue_path.name
)
print(
    "File exists:",
    dengue_path.exists()
)

Selected dengue file: dengue_monthly_proportional_2010_2024.csv
File exists: True


In [8]:
dengue = pd.read_csv(
    dengue_path,
    parse_dates=["date"]
)
weather = pd.read_csv(
    weather_path,
    parse_dates=["date"]
)
population = pd.read_csv(
    population_path,
    parse_dates=["date"]
)
print("Dengue shape:", dengue.shape)
print("Weather shape:", weather.shape)
print("Population shape:", population.shape)

Dengue shape: (4500, 5)
Weather shape: (4500, 16)
Population shape: (3300, 5)


# Standard missing value check

In [9]:
dengue_missing = (
    dengue.isna().sum()
)
dengue_missing

date                       0
year                       0
month_number               0
administrative_district    0
dengue_cases               0
dtype: int64

In [10]:
print(
    "Total dengue missing values:",
    dengue_missing.sum()
)

Total dengue missing values: 0


In [11]:
weather_missing = (
    weather.isna().sum()
)
weather_missing
print(
    "Total weather missing values:",
    weather_missing.sum()
)

Total weather missing values: 0


In [12]:
population_missing = (
    population.isna().sum()
)
population_missing
print(
    "Total population missing values:",
    population_missing.sum()
)

Total population missing values: 0


# Check temporal coverage separately

In [13]:
coverage_summary = pd.DataFrame({
    "dataset": [
        "Dengue",
        "Weather",
        "Population"
    ],
    "rows": [
        len(dengue),
        len(weather),
        len(population)
    ],
    "districts": [
        dengue["administrative_district"].nunique(),
        weather["administrative_district"].nunique(),
        population["administrative_district"].nunique()
    ],
    "months": [
        dengue["date"].nunique(),
        weather["date"].nunique(),
        population["date"].nunique()
    ],
    "first_month": [
        dengue["date"].min(),
        weather["date"].min(),
        population["date"].min()
    ],
    "last_month": [
        dengue["date"].max(),
        weather["date"].max(),
        population["date"].max()
    ],
    "standard_missing_values": [
        dengue.isna().sum().sum(),
        weather.isna().sum().sum(),
        population.isna().sum().sum()
    ]
})

coverage_summary

,dataset,rows,districts,months,first_month,last_month,standard_missing_values
0,Dengue,4500,25,180,2010-01-01,2024-12-01,0
1,Weather,4500,25,180,2010-01-01,2024-12-01,0
2,Population,3300,25,132,2014-01-01,2024-12-01,0


In [14]:
expected_months = pd.date_range(
    start="2010-01-01",
    end="2024-12-01",
    freq="MS"
)
print(
    "Expected months:",
    len(expected_months)
)
print(
    "First expected month:",
    expected_months.min()
)
print(
    "Last expected month:",
    expected_months.max()
)

Expected months: 180
First expected month: 2010-01-01 00:00:00
Last expected month: 2024-12-01 00:00:00


In [15]:
expected_districts = sorted(
    dengue[
        "administrative_district"
    ].unique()
)
print(
    "Expected districts:",
    len(expected_districts)
)
expected_districts

Expected districts: 25


['Ampara',
 'Anuradhapura',
 'Badulla',
 'Batticaloa',
 'Colombo',
 'Galle',
 'Gampaha',
 'Hambantota',
 'Jaffna',
 'Kalutara',
 'Kandy',
 'Kegalle',
 'Kilinochchi',
 'Kurunegala',
 'Mannar',
 'Matale',
 'Matara',
 'Monaragala',
 'Mullaitivu',
 'Nuwara Eliya',
 'Polonnaruwa',
 'Puttalam',
 'Ratnapura',
 'Trincomalee',
 'Vavuniya']

Create complete district month grid

In [16]:
reference_grid = pd.MultiIndex.from_product(
    [
        expected_districts,
        expected_months
    ],
    names=[
        "administrative_district",
        "date"
    ]
).to_frame(
    index=False
)
print(
    "Reference grid rows:",
    len(reference_grid)
)

Reference grid rows: 4500


Mark dataset availability

In [17]:
dengue_availability = (
    dengue[
        [
            "administrative_district",
            "date"
        ]
    ]
    .drop_duplicates()
    .assign(
        dengue_available=True
    )
)
weather_availability = (
    weather[
        [
            "administrative_district",
            "date"
        ]
    ]
    .drop_duplicates()
    .assign(
        weather_available=True
    )
)
population_availability = (
    population[
        [
            "administrative_district",
            "date"
        ]
    ]
    .drop_duplicates()
    .assign(
        population_available=True
    )
)

In [18]:
coverage_grid = (
    reference_grid
    .merge(
        dengue_availability,
        on=[
            "administrative_district",
            "date"
        ],
        how="left"
    )
    .merge(
        weather_availability,
        on=[
            "administrative_district",
            "date"
        ],
        how="left"
    )
    .merge(
        population_availability,
        on=[
            "administrative_district",
            "date"
        ],
        how="left"
    )
)

In [19]:
availability_columns = [
    "dengue_available",
    "weather_available",
    "population_available"
]
coverage_grid[
    availability_columns
] = (
    coverage_grid[
        availability_columns
    ]
    .fillna(False)
    .astype(bool)
)
coverage_grid.head()

,administrative_district,date,dengue_available,weather_available,population_available
0,Ampara,2010-01-01,True,True,False
1,Ampara,2010-02-01,True,True,False
2,Ampara,2010-03-01,True,True,False
3,Ampara,2010-04-01,True,True,False
4,Ampara,2010-05-01,True,True,False


Count unavailable district month observations

In [20]:
availability_summary = pd.DataFrame({
    "dataset": [
        "Dengue",
        "Weather",
        "Population"
    ],
    "available_district_months": [
        coverage_grid[
            "dengue_available"
        ].sum(),
        coverage_grid[
            "weather_available"
        ].sum(),
        coverage_grid[
            "population_available"
        ].sum()
    ],
    "unavailable_district_months": [
        (
            ~coverage_grid[
                "dengue_available"
            ]
        ).sum(),
        (
            ~coverage_grid[
                "weather_available"
            ]
        ).sum(),
        (
            ~coverage_grid[
                "population_available"
            ]
        ).sum()
    ]
})

availability_summary

,dataset,available_district_months,unavailable_district_months
0,Dengue,4500,0
1,Weather,4500,0
2,Population,3300,1200


Confirm exactly where population is unavailable

In [21]:
population_unavailable = (
    coverage_grid[
        ~coverage_grid[
            "population_available"
        ]
    ]
    .copy()
)
population_unavailable[
    "year"
] = (
    population_unavailable[
        "date"
    ].dt.year
)
population_unavailable_by_year = (
    population_unavailable
    .groupby("year")
    .size()
)
population_unavailable_by_year

year
2010    300
2011    300
2012    300
2013    300
dtype: int64

In [22]:
print(
    "Years with unavailable population:",
    sorted(
        population_unavailable[
            "year"
        ].unique()
    )
)

Years with unavailable population: [np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013)]


Check population missingness inside its actual coverage

In [23]:
population_coverage_period = (
    coverage_grid[
        coverage_grid[
            "date"
        ] >= pd.Timestamp(
            "2014-01-01"
        )
    ]
)
print(
    "Population unavailable within 2014–2024:",
    (
        ~population_coverage_period[
            "population_available"
        ]
    ).sum()
)

Population unavailable within 2014–2024: 0


In [24]:
weather[
    "complete_weather_month"
].value_counts(
    dropna=False
)

complete_weather_month
True     4450
False      50
Name: count, dtype: int64

In [25]:
incomplete_weather = (
    weather[
        ~weather[
            "complete_weather_month"
        ]
    ]
)
print(
    "Incomplete weather district-months:",
    len(incomplete_weather)
)
print(
    "Affected months:",
    sorted(
        incomplete_weather[
            "date"
        ].unique()
    )
)

Incomplete weather district-months: 50
Affected months: [Timestamp('2010-01-01 00:00:00'), Timestamp('2024-12-01 00:00:00')]


## Missingness Classification

The prepared datasets exhibit different forms of data availability and should not be treated as a single missing value problem.

### Dengue

The monthly dengue dataset contains all expected 25 administrative districts across all 180 months from January 2010 through December 2024. No standard missing values or missing district-month observations were identified.

### Weather

The monthly weather dataset also contains all expected 4,500 district-month observations and contains no standard missing feature values.

However, January 2010 and December 2024 have incomplete calendar-day coverage because the source weekly reporting sequence begins on 2 January 2010 and ends on 27 December 2024. These observations contain values and are therefore not standard missing values. They are classified as incomplete boundary coverage.

### Population

The population dataset contains complete district-month coverage from January 2014 through December 2024, with no internal missing observations during that period.

Population data are unavailable for all districts during 2010–2013. This produces 1,200 unavailable district month observations relative to the full 2010–2024 reference grid.

Because the entire four-year period precedes the available population source coverage, these observations are classified as a structural source coverage gap rather than randomly missing values.

Therefore, the 2010–2013 population gap will not be treated as ordinary missing data or automatically reconstructed using standard imputation methods.

Create a missingness classification summary

In [26]:
missingness_classification = pd.DataFrame({
    "dataset": [
        "Dengue",
        "Weather",
        "Population"
    ],
    "standard_missing_values": [
        dengue.isna().sum().sum(),
        weather.isna().sum().sum(),
        population.isna().sum().sum()
    ],
    "missing_district_months": [
        (
            ~coverage_grid[
                "dengue_available"
            ]
        ).sum(),

        (
            ~coverage_grid[
                "weather_available"
            ]
        ).sum(),

        (
            ~coverage_grid[
                "population_available"
            ]
        ).sum()
    ],

    "special_coverage_issue": [
        "None",
        "Partial boundary coverage",
        "Structural pre-2014 coverage gap"
    ]
})

missingness_classification

,dataset,standard_missing_values,missing_district_months,special_coverage_issue
0,Dengue,0,0,None
1,Weather,0,0,Partial boundary coverage
2,Population,0,1200,Structural pre-2014 coverage gap


## Variables for Controlled Missing-Value Experiments

The validated prepared datasets do not contain ordinary internal missing values suitable for directly comparing imputation algorithms.

Therefore, the imputation method comparison will use controlled artificial masking of observed values.

Known observed values will be temporarily hidden, reconstructed using candidate imputation methods and compared with their original values. This allows reconstruction error to be measured objectively because the true values are known.

The primary imputation experiment will focus on continuous environmental variables, including:

- mean temperature;
- mean humidity;
- total rainfall;
- mean wind speed.

Candidate methods will include:

1. linear interpolation;
2. seasonal reconstruction;
3. K-nearest-neighbour (KNN) imputation.

The structural population gap for 2010–2013 and the incomplete weather boundary months will not be used as artificial ground truth because their true unobserved values are not available.

Add weather completeness to the reference grid

In [27]:
weather_completeness = (
    weather[
        [
            "administrative_district",
            "date",
            "complete_weather_month"
        ]
    ]
    .copy()
)
coverage_grid = (
    coverage_grid
    .merge(
        weather_completeness,
        on=[
            "administrative_district",
            "date"
        ],
        how="left"
    )
)
coverage_grid.head()

,administrative_district,date,dengue_available,weather_available,population_available,complete_weather_month
0,Ampara,2010-01-01,True,True,False,False
1,Ampara,2010-02-01,True,True,False,True
2,Ampara,2010-03-01,True,True,False,True
3,Ampara,2010-04-01,True,True,False,True
4,Ampara,2010-05-01,True,True,False,True


In [28]:
print(
    "Complete weather district-months:",
    coverage_grid[
        "complete_weather_month"
    ].sum()
)
print(
    "Incomplete weather district-months:",
    (
        coverage_grid[
            "complete_weather_month"
        ] == False
    ).sum()
)

Complete weather district-months: 4450
Incomplete weather district-months: 50


Identify the common source overlap period

In [29]:
common_source_coverage = (
    coverage_grid[
        coverage_grid[
            [
                "dengue_available",
                "weather_available",
                "population_available"
            ]
        ].all(axis=1)
    ]
    .copy()
)
print(
    "Common-source district-months:",
    len(common_source_coverage)
)
print(
    "First common-source month:",
    common_source_coverage["date"].min()
)
print(
    "Last common-source month:",
    common_source_coverage["date"].max()
)
print(
    "Unique common-source months:",
    common_source_coverage[
        "date"
    ].nunique()
)

Common-source district-months: 3300
First common-source month: 2014-01-01 00:00:00
Last common-source month: 2024-12-01 00:00:00
Unique common-source months: 132


Apply strict weather-completeness criterion

In [30]:
strict_complete_coverage = (
    coverage_grid[
        (
            coverage_grid[
                "dengue_available"
            ]
        )
        &
        (
            coverage_grid[
                "weather_available"
            ]
        )
        &
        (
            coverage_grid[
                "population_available"
            ]
        )
        &
        (
            coverage_grid[
                "complete_weather_month"
            ] == True
        )
    ]
    .copy()
)

print(
    "Strict complete district-months:",
    len(strict_complete_coverage)
)

print(
    "Strict complete months:",
    strict_complete_coverage[
        "date"
    ].nunique()
)

print(
    "First strict month:",
    strict_complete_coverage[
        "date"
    ].min()
)

print(
    "Last strict month:",
    strict_complete_coverage[
        "date"
    ].max()
)

Strict complete district-months: 3275
Strict complete months: 131
First strict month: 2014-01-01 00:00:00
Last strict month: 2024-11-01 00:00:00


In [31]:
coverage_analysis = pd.DataFrame({
    "coverage_definition": [
        "Full dengue reference grid",
        "Dengue observations available",
        "Weather observations available",
        "Population observations available",
        "All three sources available",
        "All three sources + complete weather coverage"
    ],

    "district_months": [
        len(reference_grid),
        coverage_grid[
            "dengue_available"
        ].sum(),
        coverage_grid[
            "weather_available"
        ].sum(),
        coverage_grid[
            "population_available"
        ].sum(),
        len(common_source_coverage),
        len(strict_complete_coverage)
    ]
})

coverage_analysis

,coverage_definition,district_months
0,Full dengue reference grid,4500
1,Dengue observations available,4500
2,Weather observations available,4500
3,Population observations available,3300
4,All three sources available,3300
5,All three sources + complete weather coverage,3275


Quantify percentages

In [32]:
total_reference_rows = len(
    reference_grid
)
population_coverage_percent = (
    coverage_grid[
        "population_available"
    ].sum()
    /
    total_reference_rows
    * 100
)
incomplete_weather_percent = (
    len(incomplete_weather)
    /
    total_reference_rows
    * 100
)
print(
    "Population coverage of full reference grid:",
    round(
        population_coverage_percent,
        2
    ),
    "%"
)
print(
    "Incomplete weather observations:",
    round(
        incomplete_weather_percent,
        2
    ),
    "%"
)

Population coverage of full reference grid: 73.33 %
Incomplete weather observations: 1.11 %


Define weather variables eligible for Notebook 08

In [33]:
imputation_experiment_variables = [
    "temperature_mean_c",
    "humidity_mean_percent",
    "rainfall_total_mm",
    "wind_speed_mean_kmh"
]
for variable in imputation_experiment_variables:

    print(
        variable,
        "missing =",
        weather[variable]
        .isna()
        .sum()
    )

temperature_mean_c missing = 0
humidity_mean_percent missing = 0
rainfall_total_mm missing = 0
wind_speed_mean_kmh missing = 0


In [34]:
weather_imputation_base = (
    weather[
        weather[
            "complete_weather_month"
        ] == True
    ]
    .copy()
)
print(
    "Weather observations available for "
    "controlled masking:",
    len(weather_imputation_base)
)

print(
    "Districts:",
    weather_imputation_base[
        "administrative_district"
    ].nunique()
)

print(
    "First month:",
    weather_imputation_base[
        "date"
    ].min()
)

print(
    "Last month:",
    weather_imputation_base[
        "date"
    ].max()
)

Weather observations available for controlled masking: 4450
Districts: 25
First month: 2010-02-01 00:00:00
Last month: 2024-11-01 00:00:00


## Controlled Imputation Experiment Eligibility

The missing value analysis shows that the prepared weather variables contain no naturally occurring internal missing values. Therefore, direct evaluation of imputation accuracy on naturally missing observations is not possible because their true values would be unknown.

Controlled masking will instead be performed on known weather observations.

The experiment will initially evaluate four continuous environmental variables:

- mean temperature;
- mean humidity;
- total rainfall;
- mean wind speed.

January 2010 and December 2024 are excluded from the controlled masking pool because these months already have incomplete calendar-day coverage. They remain in the prepared weather dataset but are not appropriate ground-truth observations for evaluating artificial missing-value reconstruction.

Population observations from 2010–2013 are also excluded from controlled imputation experiments because no true source values are available for evaluating reconstruction accuracy.

The controlled experiment will compare linear interpolation, seasonal reconstruction, and K-nearest-neighbour (KNN) imputation using artificially masked observations whose original values are known.

## Artificial Missingness Design

A single random missingness experiment may not adequately represent realistic temporal data gaps.

Therefore, the subsequent imputation experiment will evaluate more than one artificial missingness pattern.

Two masking scenarios will be considered:

1. **Random masking** — individual observed values are randomly hidden to simulate isolated missing observations.

2. **Contiguous temporal masking** — consecutive monthly observations are hidden within a district time series to simulate short temporal gaps.

Multiple masking proportions and/or gap lengths will be evaluated where computationally practical.

The same masked observations will be supplied to each candidate imputation method so that Linear, Seasonal and KNN reconstruction errors can be compared fairly.

Original unmasked observations will be retained separately as ground truth and will never be overwritten.

In [35]:
RESULTS_DIR = (
    PROJECT_ROOT /
    "results" /
    "tables"
)
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)
print(
    "Results directory:",
    RESULTS_DIR
)

Results directory: d:\Dengue_Project\results\tables


In [36]:
coverage_summary.to_csv(
    RESULTS_DIR /
    "missing_value_coverage_summary.csv",
    index=False
)
availability_summary.to_csv(
    RESULTS_DIR /
    "dataset_availability_summary.csv",
    index=False
)
missingness_classification.to_csv(
    RESULTS_DIR /
    "missingness_classification.csv",
    index=False
)
coverage_analysis.to_csv(
    RESULTS_DIR /
    "coverage_analysis.csv",
    index=False
)
print(
    "Missing value analysis tables saved."
)

Missing value analysis tables saved.


In [37]:
coverage_summary.to_csv(
    RESULTS_DIR /
    "missing_value_coverage_summary.csv",
    index=False
)
availability_summary.to_csv(
    RESULTS_DIR /
    "dataset_availability_summary.csv",
    index=False
)
missingness_classification.to_csv(
    RESULTS_DIR /
    "missingness_classification.csv",
    index=False
)
coverage_analysis.to_csv(
    RESULTS_DIR /
    "coverage_analysis.csv",
    index=False
)
print(
    "Missing-value analysis tables saved."
)

Missing-value analysis tables saved.


## Key Findings

1. The prepared dengue, weather and population datasets contain no standard missing values within their respective source coverage periods.

2. Dengue provides complete coverage for all 25 administrative districts across 180 months from January 2010 through December 2024, giving 4,500 district month observations.

3. Weather also provides 4,500 district month observations with no missing feature values. However, 50 district month observations corresponding to January 2010 and December 2024 have incomplete calendar day coverage.

4. Population provides complete coverage for all 25 administrative districts from January 2014 through December 2024, giving 3,300 district month observations.

5. Relative to the complete 2010–2024 reference grid, population has 1,200 unavailable district month observations. All occur during 2010–2013.

6. No population observations are missing internally within the available 2014–2024 source period.

7. The 2010–2013 population gap is classified as a structural source coverage limitation rather than ordinary missingness and will not be automatically filled using standard imputation methods.

8. All three data sources overlap for 3,300 district month observations from January 2014 through December 2024.

9. Under a strict requirement of complete weather calendar coverage, 3,275 district month observations remain, covering January 2014 through November 2024.

10. Because the environmental variables contain no natural internal missing values, controlled artificial masking will be used to compare imputation methods objectively.

11. Mean temperature, mean humidity, total rainfall, and mean wind speed are selected as the primary environmental variables for the controlled imputation experiment.

12. Linear interpolation, seasonal reconstruction and KNN imputation will be evaluated using identical artificial missingness patterns and known original values as ground truth.

## Conclusion

Missing value and temporal coverage analysis was completed for the prepared dengue, weather and population datasets.

No standard missing values were identified within the available observations of any dataset. Dengue and weather provide complete district month records from 2010 through 2024, while population data are available only from 2014 through 2024.

The absence of population observations during 2010–2013 is a structural source-coverage limitation rather than an internal missing-value problem. Consequently, these values will not be automatically reconstructed using ordinary imputation methods.

The weather dataset contains values for every district-month, but January 2010 and December 2024 have incomplete calendar day coverage because of the boundaries of the original weekly reporting sequence. These observations are explicitly distinguished from standard missing values.

Since the validated environmental variables contain no naturally occurring internal missing observations, subsequent comparison of imputation methods will use controlled artificial masking. Known weather values will be temporarily hidden and reconstructed using Linear, Seasonal and KNN methods. Reconstruction performance can then be measured against the retained ground truth observations.

This analysis establishes the missingness framework required for the subsequent imputation experiments and prevents structural coverage limitations from being incorrectly treated as ordinary missing data.